# 🔬 BioHub Cell Tracking — Kaggle Submission Notebook

**Competition:** [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

**Score formula:** `edge_jaccard + 0.1 × division_jaccard`

### Pipeline overview
1. **Segmentation** — Cellpose 3D (cyto3 model) with blob-detector fallback
2. **Post-processing** — size filter (50–50k voxels) + relabelling
3. **Tracking** — Hungarian linker (max 10 µm, volume-cost enabled) + gap-2 bridging
4. **Division detection** — orphan-pair volume-consistency classifier
5. **Submission** — 10-column CSV (nodes + edges)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# On Kaggle these are usually pre-installed; uncomment if needed.
# !pip install -q cellpose>=3.0.0 scikit-image>=0.21 scipy zarr>=3.0 tqdm
import sys, subprocess
print(f'Python {sys.version}')

In [ ]:
# ── Cell 2: Install the biohub_tracking package ───────────────────────────────
# Option A: this notebook is inside the repository (recommended on Kaggle
# after adding the GitHub repo as a dataset).
import os, pathlib

# Adjust REPO_ROOT if needed
REPO_ROOT = pathlib.Path("/kaggle/input/biohub-cell-tracking-code")  # dataset with repo
if not REPO_ROOT.exists():
    # Fallback: assume the notebook is already inside the repo
    REPO_ROOT = pathlib.Path("..").resolve()

if (REPO_ROOT / "setup.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT), "-q"],
                   check=True)
    print(f"Installed biohub_tracking from {REPO_ROOT}")
else:
    print("WARNING: setup.py not found — make sure biohub_tracking is importable.")

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
import logging
import pathlib

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler()],
)

# ── Paths ──
TEST_DIR  = pathlib.Path("/kaggle/input/biohub-cell-tracking-during-development/test")
OUTPUT    = pathlib.Path("/kaggle/working/submission.csv")

print(f"Test directory : {TEST_DIR}")
print(f"Output CSV     : {OUTPUT}")

zarr_files = sorted(TEST_DIR.glob("*.zarr"))
print(f"Found {len(zarr_files)} test samples:")
for p in zarr_files[:5]:
    print(f"  {p.name}")
if len(zarr_files) > 5:
    print(f"  ... and {len(zarr_files) - 5} more.")

In [ ]:
# ── Cell 4: Inspect one sample ────────────────────────────────────────────────
if zarr_files:
    import zarr, numpy as np
    sample = zarr_files[0]
    root = zarr.open(str(sample), mode="r")
    arr  = root["0"] if "0" in root else root
    print(f"Sample : {sample.name}")
    print(f"Shape  : {arr.shape}  (T, Z, Y, X)")
    print(f"dtype  : {arr.dtype}")
    print(f"Chunks : {arr.chunks}")

In [ ]:
# ── Cell 5: Run full pipeline & generate submission ───────────────────────────
# This imports the fixed pipeline from submission.py (in the repo root).
import importlib.util, sys as _sys

# Dynamically load submission.py so we don't need it on sys.path
_spec = importlib.util.spec_from_file_location(
    "submission_module",
    str(REPO_ROOT / "submission.py"),
)
_mod = importlib.util.module_from_spec(_spec)
_sys.modules["submission_module"] = _mod
_spec.loader.exec_module(_mod)

# Run the pipeline
_mod.generate_submission(TEST_DIR, OUTPUT)
print(f"\n✅ submission.csv written to {OUTPUT}")

In [ ]:
# ── Cell 6: Validate and preview submission ───────────────────────────────────
import pandas as pd

df = pd.read_csv(OUTPUT)

print("=" * 60)
print(f"Total rows   : {len(df):,}")
print(f"Node rows    : {(df.row_type == 'node').sum():,}")
print(f"Edge rows    : {(df.row_type == 'edge').sum():,}")
print(f"Datasets     : {df.dataset.nunique()}")
print(f"Columns      : {list(df.columns)}")
print("=" * 60)

# Validation checks
assert list(df.columns) == ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"], \
    "Column names do not match expected schema!"
assert df.row_type.isin(["node", "edge"]).all(), "Unexpected row_type values!"

nodes = df[df.row_type == "node"]
edges = df[df.row_type == "edge"]

# Nodes: node_id > 0, t >= 0, coordinates != -1
assert (nodes.node_id > 0).all(), "Node rows must have positive node_id"
assert (nodes.t >= 0).all(), "Node rows must have t >= 0"

# Edges: source_id & target_id must exist as node_ids
all_node_ids = set(nodes.node_id)
assert edges.source_id.isin(all_node_ids).all(), "Edge source_id references unknown node"
assert edges.target_id.isin(all_node_ids).all(), "Edge target_id references unknown node"

print("\n✅ All validation checks passed!")
print("\nFirst 10 rows:")
df.head(10)

In [ ]:
# ── Cell 7: Per-dataset statistics ───────────────────────────────────────────
stats = df.groupby(["dataset", "row_type"]).size().unstack(fill_value=0)
stats["edge_ratio"] = (stats.get("edge", 0) / stats.get("node", 1)).round(2)
print(stats.to_string())

## 📤 Submission

Once the cells above run without errors, go to the **Output** panel in this Kaggle notebook and click **Submit** (top-right) to submit `submission.csv` to the competition leaderboard.

---

## 🛠️ Tuning Tips

| Parameter | Location | Effect |
|-----------|----------|--------|
| `LINK_MAX_DIST_UM` | `submission.py` | Larger → more links (reduce FN but increase FP) |
| `MAX_FRAME_GAP` | `submission.py` | `2` allows 1-frame blinks; set higher for slower-moving cells |
| `min_size` | `CellSegmenter` | Increase to suppress debris |
| `diameter` | `CellSegmenter` | Match to typical cell diameter in pixels |
| `max_distance_um` | `DivisionClassifier` | How far daughters can be from their mother |

The **single biggest score gain** comes from having well-calibrated Cellpose 3D segmentation. If `cellpose` isn't available on Kaggle's environment, install it with:
```bash
!pip install cellpose>=3.0.0
```